# SpaceLaunch Analytics — Exploratory Data Analysis

**Project:** ETL + EDA Pipeline  
**Dataset:** Validated historical rocket-launch records (2020–2025)

This notebook answers the six research questions in the project's *Data Visualization Plan*.
It reads the validated `launches` table from SQLite. It does not call the API or modify the database.

> **Expected location:** Save this notebook inside the project's `notebooks/` folder.  
> Expected database: `data/database/launches.db`  
> Expected validation report: `data/processed/validation_report.json`

## 1. Import libraries and configure paths

The notebook uses:
- **pandas** for data analysis
- **Plotly** for interactive charts
- **sqlite3** to read the SQLite database
- **matplotlib** only if needed for supplementary static output

In [ ]:
from pathlib import Path
import sqlite3
import json
import warnings

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display, Markdown

warnings.filterwarnings("ignore")

# This notebook is expected to be inside the project's "notebooks" folder.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

DB_PATH = PROJECT_ROOT / "data" / "database" / "launches.db"
VALIDATION_REPORT_PATH = PROJECT_ROOT / "data" / "processed" / "validation_report.json"
REJECTED_RECORDS_PATH = PROJECT_ROOT / "data" / "processed" / "rejected_records.json"

print("Project folder:", PROJECT_ROOT)
print("Database found:", DB_PATH.exists())
print("Validation report found:", VALIDATION_REPORT_PATH.exists())

## 2. Load validated data from SQLite

The database schema follows `load.py`. We read all rows from the `launches` table into a pandas DataFrame.

In [ ]:
if not DB_PATH.exists():
    raise FileNotFoundError(
        f"Database not found: {DB_PATH}\n"
        "Make sure you have run src/load.py and that this notebook is inside the notebooks folder."
    )

with sqlite3.connect(DB_PATH) as connection:
    df = pd.read_sql_query("SELECT * FROM launches", connection)

if df.empty:
    raise ValueError("The launches table is empty. Run the ETL pipeline before EDA.")

# Ensure fields have the expected data types.
df["launch_datetime"] = pd.to_datetime(df["launch_datetime"], errors="coerce", utc=True)
df["launch_year"] = pd.to_numeric(df["launch_year"], errors="coerce").astype("Int64")
df["launch_month"] = pd.to_numeric(df["launch_month"], errors="coerce").astype("Int64")

for column in ["launch_name", "provider", "rocket", "launch_location", "country", "status"]:
    if column in df.columns:
        df[column] = df[column].fillna("Unknown").astype(str).str.strip()
        df.loc[df[column].eq(""), column] = "Unknown"

print(f"Validated launch records loaded: {len(df):,}")
display(df.head())

## 3. Dataset overview

A quick overview of the dataset, its date coverage, and the number of distinct providers, rockets, and launch locations.

In [ ]:
def distinct_known(column):
    """Distinct values in a column, not counting the 'Unknown' placeholder for missing data."""
    return df.loc[df[column].ne("Unknown"), column].nunique()


overview = pd.DataFrame({
    "Metric": [
        "Total validated launches",
        "First recorded launch date",
        "Last recorded launch date",
        "Distinct providers",
        "Distinct rocket configurations",
        "Distinct launch locations",
        "Countries represented"
    ],
    "Value": [
        f"{len(df):,}",
        df["launch_datetime"].min().strftime("%Y-%m-%d") if df["launch_datetime"].notna().any() else "Not available",
        df["launch_datetime"].max().strftime("%Y-%m-%d") if df["launch_datetime"].notna().any() else "Not available",
        distinct_known("provider"),
        distinct_known("rocket"),
        distinct_known("launch_location"),
        distinct_known("country")
    ]
})

display(overview)
display(df.describe(include="all").T)

# Research Question 1 — Launch activity over time

**Question:** How has the number of recorded launches changed over time, by year and month?

The monthly line chart shows rises and dips over time. The yearly bar chart provides a high-level comparison.

In [ ]:
# Monthly launch counts, ordered chronologically.
monthly = (
    df.dropna(subset=["launch_datetime"])
      .assign(month_start=lambda x: x["launch_datetime"].dt.tz_localize(None).dt.to_period("M").dt.to_timestamp())
      .groupby("month_start")
      .size()
)

# Include months with zero launches so a gap is shown as 0, not skipped.
all_months = pd.date_range(monthly.index.min(), monthly.index.max(), freq="MS")
monthly = (
    monthly.reindex(all_months, fill_value=0)
           .rename_axis("month_start")
           .reset_index(name="launches")
)

fig = px.line(
    monthly,
    x="month_start",
    y="launches",
    markers=True,
    title="Monthly Recorded Launches",
    labels={"month_start": "Month", "launches": "Number of launches"}
)
fig.update_layout(hovermode="x unified")
fig.show()

yearly = (
    df.dropna(subset=["launch_year"])
      .groupby("launch_year")
      .size()
      .reset_index(name="launches")
      .sort_values("launch_year")
)
yearly["launch_year"] = yearly["launch_year"].astype(str)

fig = px.bar(
    yearly,
    x="launch_year",
    y="launches",
    text="launches",
    title="Total Recorded Launches by Year",
    labels={"launch_year": "Year", "launches": "Number of launches"}
)
fig.update_traces(textposition="outside")
fig.show()

# Research Question 2 — Most frequent providers, rockets, and locations

**Question:** Which launch service providers, rocket configurations, and launch locations appear most frequently?

The charts show the top 10 in each category. Remaining categories are combined into **Other** when there are more than 10.

In [ ]:
def top_ten_with_other(data, column):
    counts = data[column].replace("", "Unknown").fillna("Unknown").value_counts()
    top = counts.head(10)
    if len(counts) > 10:
        top.loc["Other"] = counts.iloc[10:].sum()
    return top.sort_values(ascending=True).rename_axis(column).reset_index(name="launches")

for column, title in [
    ("provider", "Top 10 Launch Service Providers"),
    ("rocket", "Top 10 Rocket Configurations"),
    ("launch_location", "Top 10 Launch Locations")
]:
    chart_data = top_ten_with_other(df, column)
    fig = px.bar(
        chart_data,
        x="launches",
        y=column,
        orientation="h",
        text="launches",
        title=title,
        labels={column: column.replace("_", " ").title(), "launches": "Number of launches"}
    )
    fig.update_traces(textposition="outside")
    fig.show()

# Research Question 3 — Launch outcomes across providers and years

**Question:** How do recorded launch outcomes vary across providers and years?

These are 100% stacked bar charts. Each bar represents the outcome distribution within one provider or year.
Only records with a known status are included. Non-final statuses (for example, To Be Confirmed or On Hold) remain visible as **Other known outcomes** rather than being treated as failures.

In [ ]:
OUTCOME_COLORS = {
    "Success": "#2ca02c",
    "Partial failure": "#ff9f1c",
    "Failure": "#d62728",
    "Other known outcomes": "#9aa0a6",
}
OUTCOME_ORDER = list(OUTCOME_COLORS)


def outcome_group(status):
    value = str(status).strip().casefold()
    if value == "launch successful":
        return "Success"
    if value == "launch failure":
        return "Failure"
    if value in {"launch was a partial failure", "launch partial failure"}:
        return "Partial failure"
    return "Other known outcomes"

outcomes = df.copy()
outcomes["outcome_group"] = outcomes["status"].apply(outcome_group)
outcomes = outcomes[outcomes["status"].ne("Unknown")]

# Show the providers with the most records so the chart remains readable.
top_providers = outcomes["provider"].value_counts().head(10).index
provider_outcomes = outcomes[outcomes["provider"].isin(top_providers)]

provider_mix = (
    provider_outcomes.groupby(["provider", "outcome_group"])
    .size().rename("launches").reset_index()
)
provider_mix["provider_total"] = provider_mix.groupby("provider")["launches"].transform("sum")
provider_mix["share"] = provider_mix["launches"] / provider_mix["provider_total"] * 100

fig = px.bar(
    provider_mix,
    x="share",
    y="provider",
    color="outcome_group",
    orientation="h",
    barmode="stack",
    title="Launch Outcome Distribution by Top Providers (100%)",
    labels={"share": "Share of launches (%)", "provider": "Provider", "outcome_group": "Recorded outcome"},
    custom_data=["launches", "provider_total"],
    color_discrete_map=OUTCOME_COLORS,
    category_orders={"outcome_group": OUTCOME_ORDER}
)
fig.update_traces(
    hovertemplate="%{y}<br>Outcome: %{fullData.name}<br>Share: %{x:.1f}%<br>Count: %{customdata[0]} of %{customdata[1]}<extra></extra>"
)
fig.update_layout(xaxis_range=[0, 100])
fig.show()

year_outcomes = outcomes.dropna(subset=["launch_year"]).copy()
year_outcomes["launch_year"] = year_outcomes["launch_year"].astype(str)
year_mix = (
    year_outcomes.groupby(["launch_year", "outcome_group"])
    .size().rename("launches").reset_index()
)
year_mix["year_total"] = year_mix.groupby("launch_year")["launches"].transform("sum")
year_mix["share"] = year_mix["launches"] / year_mix["year_total"] * 100

fig = px.bar(
    year_mix,
    x="share",
    y="launch_year",
    color="outcome_group",
    orientation="h",
    barmode="stack",
    title="Launch Outcome Distribution by Year (100%)",
    labels={"share": "Share of launches (%)", "launch_year": "Year", "outcome_group": "Recorded outcome"},
    custom_data=["launches", "year_total"],
    color_discrete_map=OUTCOME_COLORS,
    category_orders={"outcome_group": OUTCOME_ORDER}
)
fig.update_traces(
    hovertemplate="%{y}<br>Outcome: %{fullData.name}<br>Share: %{x:.1f}%<br>Count: %{customdata[0]} of %{customdata[1]}<extra></extra>"
)
fig.update_layout(xaxis_range=[0, 100])
fig.show()

# Research Question 4 — Year-over-year changes and monthly patterns

**Question:** What year-over-year changes and monthly patterns can be observed in launch activity?

The heatmap compares each calendar month across years. The second chart shows the percentage change in annual launch totals compared with the previous year. The first year has no previous year for comparison.

In [ ]:
year_month = (
    df.dropna(subset=["launch_year", "launch_month"])
      .groupby(["launch_year", "launch_month"])
      .size()
      .rename("launches")
      .reset_index()
)

heatmap_data = year_month.pivot(
    index="launch_year", columns="launch_month", values="launches"
).reindex(columns=range(1, 13)).fillna(0)

month_names = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
               "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

fig = px.imshow(
    heatmap_data,
    labels={"x": "Month", "y": "Year", "color": "Number of launches"},
    x=month_names,
    y=heatmap_data.index.astype(str),
    text_auto=True,
    aspect="auto",
    title="Launch Activity Heatmap: Year × Month",
    color_continuous_scale="Blues"
)
fig.show()

annual_counts = (
    df.dropna(subset=["launch_year"])
      .groupby("launch_year").size()
      .rename("launches").reset_index()
      .sort_values("launch_year")
)
annual_counts["yoy_change_pct"] = annual_counts["launches"].pct_change() * 100
yoy = annual_counts.dropna(subset=["yoy_change_pct"]).copy()
yoy["direction"] = np.where(yoy["yoy_change_pct"] >= 0, "Increase", "Decrease")
yoy["launch_year"] = yoy["launch_year"].astype(str)

fig = px.bar(
    yoy,
    x="launch_year",
    y="yoy_change_pct",
    color="direction",
    text=yoy["yoy_change_pct"].map(lambda value: f"{value:+.1f}%"),
    title="Year-over-Year Change in Recorded Launches",
    labels={"launch_year": "Year", "yoy_change_pct": "Change from previous year (%)", "direction": "Change"}
)
fig.add_hline(y=0, line_dash="dash", line_color="gray")
fig.update_traces(textposition="outside")
fig.show()

display(annual_counts)

# Research Question 5 — Data quality of source records

**Question:** What proportion of source records contain missing, inconsistent, invalid, or duplicate data?

The validation summary compares total extracted records with valid and rejected records. Rejection categories are shown separately as missing, invalid, and duplicate records.

In [ ]:
def load_json_if_available(path):
    if path.exists():
        with open(path, "r", encoding="utf-8") as file:
            return json.load(file)
    return None

validation_report = load_json_if_available(VALIDATION_REPORT_PATH)

if validation_report is None:
    print("Validation report not found:", VALIDATION_REPORT_PATH)
    print("Q5 cannot be fully calculated from the SQLite table alone.")
    print("Place data/processed/validation_report.json in the expected location and rerun this section.")
else:
    print("Validation report loaded. Raw report structure:")
    print(json.dumps(validation_report, indent=4, ensure_ascii=False))

### Q5A. Validation summary and KPI overview

This section reads the exact validation-report keys to calculate record counts and quality KPIs. It also displays validation outcomes as separate categories.

In [ ]:
if validation_report is not None:

    # Read counts directly from validation_report.json
    total_extracted = validation_report["total_records"]
    valid_count = validation_report["valid_records"]
    rejected_count = validation_report["rejected_records"]
    duplicates = validation_report["duplicate_records"]
    missing_count = validation_report["missing_values"]
    invalid_count = validation_report["invalid_records"]

    # Records remaining after duplicate removal
    after_dedup = total_extracted - duplicates

    # Validation summary
    quality_metrics = pd.DataFrame({
        "Metric": [
            "Total extracted",
            "After duplicate removal",
            "Valid records",
            "Rejected records",
            "Duplicate records"
        ],
        "Count": [
            total_extracted,
            after_dedup,
            valid_count,
            rejected_count,
            duplicates
        ]
    })

    display(quality_metrics)

    # KPI percentages
    if total_extracted > 0:

        kpis = pd.DataFrame({
            "KPI": [
                "Valid records (%)",
                "Rejected records (%)",
                "Duplicate records (%)"
            ],
            "Value": [
                f"{valid_count / total_extracted * 100:.2f}%",
                f"{rejected_count / total_extracted * 100:.2f}%",
                f"{duplicates / total_extracted * 100:.2f}%"
            ]
        })

        display(kpis)

    # Validation outcomes — separate categories, not a sequential funnel
    outcome_df = pd.DataFrame({
        "Category": [
            "Valid",
            "Missing",
            "Invalid",
            "Duplicate"
        ],
        "Records": [
            valid_count,
            missing_count,
            invalid_count,
            duplicates
        ]
    })

    fig = px.bar(
        outcome_df,
        x="Category",
        y="Records",
        title="Validation Results by Category",
        text="Records",
        labels={
            "Category": "Validation Category",
            "Records": "Number of Records"
        }
    )

    fig.update_traces(textposition="outside")
    fig.show()

    # Verify record accounting
    if valid_count + rejected_count == total_extracted:
        print("Record accounting verified successfully.")
    else:
        print("Warning: Record accounting does not match!")

else:
    print("Validation report is unavailable. Q5 cannot be calculated.")

### Q5B. Issue types and field completeness

This section uses rejection categories from the validation report directly. It also calculates completeness for the validated records in SQLite. Completeness in the validated table is not a substitute for source-record missingness, because invalid rows have already been rejected.

In [ ]:
# Q5B: Validation issue types and field completeness

# ------------------------------------------
# 1. Validation issues by type
# ------------------------------------------

if validation_report is not None:

    # Read counts directly from validation_report.json
    issue_counts = {
        "Missing records": validation_report["missing_values"],
        "Invalid records": validation_report["invalid_records"],
        "Duplicate records": validation_report["duplicate_records"]
    }

    issue_df = pd.DataFrame(
        list(issue_counts.items()),
        columns=["Issue type", "Records"]
    ).sort_values("Records", ascending=True)

    # Display issue counts
    display(issue_df)

    # Plot validation issues
    fig = px.bar(
        issue_df,
        x="Records",
        y="Issue type",
        orientation="h",
        text="Records",
        title="Validation Issues by Type",
        labels={
            "Records": "Number of Records",
            "Issue type": "Validation Issue"
        }
    )

    fig.update_traces(textposition="outside")
    fig.show()

else:
    print("Validation report is unavailable. Issue types cannot be calculated.")


# ------------------------------------------
# 2. Field completeness in validated records
# ------------------------------------------

completeness = (
    df.replace({"": np.nan, "Unknown": np.nan})
      .notna()
      .mean()
      .mul(100)
      .sort_values()
      .rename("Completeness (%)")
      .rename_axis("Field")
      .reset_index()
)

# Plot field completeness
fig = px.bar(
    completeness,
    x="Completeness (%)",
    y="Field",
    orientation="h",
    text=completeness["Completeness (%)"].map(
        lambda value: f"{value:.1f}%"
    ),
    title="Field Completeness in Validated Database Records",
    labels={
        "Completeness (%)": "Completeness (%)",
        "Field": "Field"
    }
)

fig.update_xaxes(range=[0, 100])
fig.update_traces(textposition="outside")
fig.show()

# Display completeness values
display(
    completeness.sort_values("Completeness (%)")
)

# Research Question 6 — Providers with the highest recorded success rates

**Question:** Which providers have the highest recorded success rates among launches with known final outcomes?

Success rate is calculated as:

\[
\text{Success Rate} = \frac{\text{Successful launches}}{\text{Successful + failed + partial-failure launches}} \times 100
\]

Pending, unknown, and other non-final statuses are excluded from the denominator. A minimum launch-count threshold is used to avoid ranking providers based on very small samples.

In [ ]:
FINAL_SUCCESS = "Launch Successful"
FINAL_FAILURE = "Launch Failure"
FINAL_PARTIAL_FAILURE = "Launch was a Partial Failure"
FINAL_STATUSES = {FINAL_SUCCESS, FINAL_FAILURE, FINAL_PARTIAL_FAILURE}

final_outcomes = df[df["status"].isin(FINAL_STATUSES)].copy()

provider_performance = (
    final_outcomes.assign(is_success=final_outcomes["status"].eq(FINAL_SUCCESS))
    .groupby("provider")
    .agg(
        known_final_launches=("launch_id", "count"),
        successful_launches=("is_success", "sum")
    )
    .reset_index()
)

provider_performance["success_rate_pct"] = (
    provider_performance["successful_launches"]
    / provider_performance["known_final_launches"] * 100
)

# Adjust this threshold if your report requires a different minimum sample size.
MIN_FINAL_LAUNCHES = 5

ranked_providers = (
    provider_performance[
        provider_performance["known_final_launches"] >= MIN_FINAL_LAUNCHES
    ]
    .sort_values(["success_rate_pct", "known_final_launches"], ascending=True)
)

if ranked_providers.empty:
    print(
        f"No provider has at least {MIN_FINAL_LAUNCHES} launches with known final outcomes. "
        "Lower MIN_FINAL_LAUNCHES if you want to inspect smaller samples."
    )
else:
    fig = px.bar(
        ranked_providers,
        x="success_rate_pct",
        y="provider",
        orientation="h",
        text=ranked_providers.apply(
            lambda row: f'{row["success_rate_pct"]:.1f}% (n={int(row["known_final_launches"])})',
            axis=1
        ),
        title=f"Provider Success Rates (Minimum {MIN_FINAL_LAUNCHES} Final Outcomes)",
        labels={
            "success_rate_pct": "Success rate (%)",
            "provider": "Provider"
        },
        hover_data={
            "successful_launches": True,
            "known_final_launches": True,
            "success_rate_pct": ":.2f"
        }
    )
    fig.update_xaxes(range=[0, 100])
    fig.update_traces(textposition="outside")
    fig.show()

display(provider_performance.sort_values(["success_rate_pct", "known_final_launches"], ascending=False))

## 4. Key findings and interpretation

Use the outputs above to write findings based on the actual dataset. Do not assume that a larger number of recorded launches means a higher success rate, and do not interpret missing or pending outcomes as failures.

Suggested points to discuss in your project presentation:
- Which year had the most recorded launches?
- Which months appear busiest across the available years?
- Which providers, rocket configurations, and launch locations occur most often?
- Do outcome distributions differ across providers or years?
- Which providers have the highest success rates after applying the minimum sample threshold?
- What are the most common validation issues?

## Notes and limitations

1. All launch-activity and outcome charts use the validated SQLite dataset.
2. Q5 requires the separate validation report and cannot be fully reconstructed from the valid-only database.
3. Success-rate calculations include only the three final outcome statuses: successful, failure, and partial failure.
4. Pending and other non-final statuses are not counted as failures.
5. The notebook is read-only with respect to SQLite; it does not change the database.
6. The chart outputs depend on the records actually present. If the database currently contains only a small test extraction, the charts will reflect that limited sample.